In [ ]:
import sys
import os
import time
import math
import numpy as np
import yaml
import cloudpickle
sys.path.append(os.path.abspath('..'))
from drone_env.drone_sim import RoomDroneEnv
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import pybullet as p

STAGE_TO_WATCH = 0

print(f"Best Model Tracker initialized. Monitoring Stage {STAGE_TO_WATCH}...")

config_path = os.path.abspath(os.path.join('..', 'configs', 'teacher_ppo.yaml'))
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

stage_key = f"stage_{STAGE_TO_WATCH}"
stage_config = config['stages'][stage_key]

NUM_OBS = stage_config['num_obstacles']
RAND_OBS = stage_config['randomize_obstacles']
RAND_COINS = stage_config['randomize_coins']
HOVER_ONLY = stage_config.get('hover_only', False)
NUM_FIXED_COINS = stage_config.get('num_fixed_coins', 4)
RUN_NAME = stage_config['run_name']
reward_weights = config['hover_rewards'] if HOVER_ONLY else config['nav_rewards']

env_raw = RoomDroneEnv(gui=True, num_obstacles=NUM_OBS, randomize_obstacles=RAND_OBS,
                       randomize_coins=RAND_COINS, reward_weights=reward_weights,
                       hover_only=HOVER_ONLY, num_fixed_coins=NUM_FIXED_COINS)
env_vec = DummyVecEnv([lambda e=env_raw: e])

model_path = os.path.abspath(os.path.join('..', 'models', RUN_NAME, 'best_model'))
vecnorm_path = f"{model_path}_vecnormalize.pkl"
print(f"Waiting for best brain at: {model_path}.zip")

while not (os.path.exists(f"{model_path}.zip") and os.path.exists(vecnorm_path)):
    print("Waiting for the first EvalCallback to create best_model.zip and .pkl...")
    time.sleep(5)

env = VecNormalize.load(vecnorm_path, env_vec)
env.training = False
env.norm_reward = False
model = PPO.load(model_path, env=env)

obs = env.reset()
p.resetDebugVisualizerCamera(cameraDistance=6.0, cameraYaw=45, cameraPitch=-30, cameraTargetPosition=[0, 0, 1])

print(f"[{RUN_NAME}] Best Model Simulation started. Updating dynamically when records are broken...")

_shaft = None
_head_l = None
_head_r = None
_debug_text = None
_prev_trail_pos = None
episode_reward = 0.0
ep_steps = 0

def _draw_arrow(base, nose_dir, shaft_id, head_l_id, head_r_id):
    RED = [1, 0, 0]
    W = 2
    tip = np.array(base) + 0.5 * nose_dir
    up = np.array([0, 0, 1])
    perp = np.cross(nose_dir, up)
    mag = np.linalg.norm(perp)
    perp = perp / mag if mag > 0.01 else np.array([1, 0, 0])
    head_base = tip - 0.12 * nose_dir
    left_pt  = (head_base + 0.07 * perp).tolist()
    right_pt = (head_base - 0.07 * perp).tolist()
    tip = tip.tolist()
    base = list(base)
    shaft_id  = p.addUserDebugLine(base, tip,      RED, W, lifeTime=0, **({'replaceItemUniqueId': shaft_id}  if shaft_id  is not None else {}))
    head_l_id = p.addUserDebugLine(tip,  left_pt,  RED, W, lifeTime=0, **({'replaceItemUniqueId': head_l_id} if head_l_id is not None else {}))
    head_r_id = p.addUserDebugLine(tip,  right_pt, RED, W, lifeTime=0, **({'replaceItemUniqueId': head_r_id} if head_r_id is not None else {}))
    return shaft_id, head_l_id, head_r_id

while True:
    action, _states = model.predict(obs, deterministic=True)
    obs, rewards, dones, infos = env.step(action)
    episode_reward += rewards[0]
    ep_steps += 1
    time.sleep(1./240.)

    if dones[0]:
        episode_reward = 0.0
        ep_steps = 0
        _shaft = _head_l = _head_r = _debug_text = None
        _prev_trail_pos = None
        time.sleep(0.5)
        try:
            with open(vecnorm_path, 'rb') as f:
                saved = cloudpickle.load(f)
            env.obs_rms = saved.obs_rms
            env.ret_rms = saved.ret_rms
            model = PPO.load(model_path, env=env)
        except Exception:
            pass

    drone_pos, ori = p.getBasePositionAndOrientation(env_raw.drone_id)
    euler = p.getEulerFromQuaternion(ori)
    rot = np.array(p.getMatrixFromQuaternion(ori)).reshape(3, 3)
    nose_dir = rot @ np.array([0, 1, 0])

    _shaft, _head_l, _head_r = _draw_arrow(drone_pos, nose_dir, _shaft, _head_l, _head_r)

    # Cyan fading trajectory trail — one segment every 12 steps (20 Hz), fades after 4s
    if ep_steps % 12 == 0:
        if _prev_trail_pos is not None:
            p.addUserDebugLine(_prev_trail_pos, list(drone_pos), [0, 0.7, 1.0], 1, lifeTime=0)
        _prev_trail_pos = list(drone_pos)

    # Distance to target: hover→virtual target, nav→nearest coin
    if HOVER_ONLY:
        dist = np.linalg.norm(np.array(drone_pos) - np.array(env_raw.hover_target))
    elif env_raw.gold_data:
        dist = min(math.sqrt(sum((a - b) ** 2 for a, b in zip(drone_pos, gd["pos"]))) for gd in env_raw.gold_data)
    else:
        dist = 0.0

    tilt_deg = math.degrees(math.sqrt(euler[0] ** 2 + euler[1] ** 2))
    ep_seconds = ep_steps / 240.0
    text = (f"Alt:{drone_pos[2]:.2f}m  Dist:{dist:.2f}m  Tilt:{tilt_deg:.1f}deg"
            f"  T:{ep_seconds:.0f}s  Ep R:{episode_reward:.1f}")
    text_pos = [drone_pos[0], drone_pos[1], drone_pos[2] + 1.2]
    if _debug_text is None:
        _debug_text = p.addUserDebugText(text, text_pos, textColorRGB=[1, 0, 0], textSize=1.2, lifeTime=0)
    else:
        _debug_text = p.addUserDebugText(text, text_pos, textColorRGB=[1, 0, 0], textSize=1.2, lifeTime=0,
                                         replaceItemUniqueId=_debug_text)
